# Train Logistic Regression

This notebook is self-contained and no longer imports helper code from sibling files in this folder.
Update the config cell, then run the remaining cells from top to bottom.


In [1]:
# =========================
# Notebook setup and imports
# =========================
!pip install pandas
!pip install rich
!pip install -U scikit-learn

from __future__ import annotations

import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
from sklearn.callback import ProgressBar, ScoringMonitor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
# ===========================
# Optional Google Drive mount
# ===========================
#try:
#    from google.colab import drive
#except ImportError:
#    print("Google Colab not detected; skipping Drive mount.")
#else:
#    drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# ==============
# Notebook config
# ==============
@dataclass(frozen=True)
class TrainingConfig:
    data: Path
    model_out: Path
    test_size: float
    random_state: int
    preview_rows: int
    preprocessed_out: Path
    categorical_encoding: Literal["onehot", "ordinal"]
    numeric_scaler: Literal["standard", "bad"]


config = TrainingConfig(
    data=Path("WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    model_out=Path("logistic_regression_churn.pkl"),
    test_size=0.2,
    random_state=42,
    preview_rows=5,
    preprocessed_out=Path("preprocessed_training_data.csv"),
    categorical_encoding="onehot",
    numeric_scaler="standard",
)

config


TrainingConfig(data=PosixPath('WA_Fn-UseC_-Telco-Customer-Churn.csv'), model_out=PosixPath('logistic_regression_churn.pkl'), test_size=0.2, random_state=42, preview_rows=5, preprocessed_out=PosixPath('preprocessed_training_data.csv'), categorical_encoding='onehot', numeric_scaler='standard')

In [4]:
# ================================
# Teaching-purpose helper utilities
# ================================
class BadMagnitudeScaler(BaseEstimator, TransformerMixin):
    """Deliberately distort feature magnitudes to show why scaling matters."""

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("BadMagnitudeScaler expects a 2D array.")

        self.n_features_in_ = X.shape[1]
        midpoint = (self.n_features_in_ - 1) / 2
        self.scale_factors_ = np.power(
            10.0,
            np.arange(self.n_features_in_, dtype=float) - midpoint,
        )
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("BadMagnitudeScaler expects a 2D array.")
        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                "BadMagnitudeScaler saw a different number of features at transform time."
            )

        return X * self.scale_factors_

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = [f"x{i}" for i in range(self.n_features_in_)]
        return np.asarray(input_features, dtype=object)


def transformed_to_dataframe(
    preprocessor: ColumnTransformer,
    X: pd.DataFrame,
) -> pd.DataFrame:
    transformed = preprocessor.transform(X)
    if hasattr(transformed, "toarray"):
        transformed = transformed.toarray()

    return pd.DataFrame(
        transformed,
        columns=preprocessor.get_feature_names_out(),
        index=X.index,
    )


def select_teaching_preview_columns(
    preprocessed_df: pd.DataFrame,
    categorical_encoding: str,
) -> pd.DataFrame:
    preview_columns = ["num__MonthlyCharges"]
    if categorical_encoding == "onehot":
        preview_columns.extend(
            [
                "cat__Contract_Month-to-month",
                "cat__Contract_One year",
                "cat__Contract_Two year",
            ]
        )
    else:
        preview_columns.append("cat__Contract")

    return preprocessed_df[preview_columns]


def print_preprocessor_summary(
    model: Pipeline,
    numeric_features: list[str],
    categorical_features: list[str],
) -> None:
    preprocessor = model.named_steps["preprocessor"]
    numeric_pipeline = preprocessor.named_transformers_["num"]
    categorical_pipeline = preprocessor.named_transformers_["cat"]

    scaler = numeric_pipeline.named_steps["scaler"]
    encoder = categorical_pipeline.named_steps["encoder"]

    scaler_summary_dict = {"feature": numeric_features}
    mean = getattr(scaler, "mean_", None)
    scale = getattr(scaler, "scale_", None)
    scale_factors = getattr(scaler, "scale_factors_", None)

    if mean is not None:
        scaler_summary_dict["mean"] = mean
    if scale is not None:
        scaler_summary_dict["scale"] = scale
    if scale_factors is not None:
        scaler_summary_dict["multiplier"] = scale_factors

    scaler_summary = pd.DataFrame(scaler_summary_dict)
    print(f"\n{scaler.__class__.__name__} summary:")
    print(scaler_summary.to_string(index=False))

    print(f"\n{encoder.__class__.__name__} categories:")
    for feature_name, categories in zip(categorical_features, encoder.categories_):
        print(f"{feature_name}: {list(categories)}")


def show_preprocessing_preview(
    model: Pipeline,
    X_train: pd.DataFrame,
    numeric_features: list[str],
    categorical_features: list[str],
    categorical_encoding: str,
    numeric_scaler: str,
    preview_rows: int,
    preprocessed_out: Path,
) -> None:
    print_preprocessor_summary(model, numeric_features, categorical_features)

    preprocessor = model.named_steps["preprocessor"]
    preprocessed_train_df = transformed_to_dataframe(preprocessor, X_train)
    preprocessed_preview = select_teaching_preview_columns(
        preprocessed_train_df,
        categorical_encoding,
    ).head(preview_rows)

    monthly_charges_label = (
        "standard-scaled" if numeric_scaler == "standard" else "badly scaled"
    )

    if categorical_encoding == "onehot":
        print(
            f"\nPreprocessed training data preview for {monthly_charges_label} "
            f"'MonthlyCharges' and 'Contract' "
            f"(first {len(preprocessed_preview)} rows):"
        )
    else:
        print(
            f"\nPreprocessed training data preview for {monthly_charges_label} "
            f"'MonthlyCharges' and "
            f"ordinal-encoded 'Contract' (first {len(preprocessed_preview)} rows):"
        )
        contract_feature_index = categorical_features.index("Contract")
        contract_categories = (
            preprocessor.named_transformers_["cat"]
            .named_steps["encoder"]
            .categories_[contract_feature_index]
        )
        contract_mapping = {
            category: index for index, category in enumerate(contract_categories)
        }
        print(f"Ordinal mapping used for Contract: {contract_mapping}")

    print(preprocessed_preview.to_string())
    preprocessed_train_df.to_csv(preprocessed_out, index=False)
    print(f"Saved preprocessed training data to: {preprocessed_out}")


In [ ]:
# =====================
# Core training helpers
# =====================
def load_data(csv_path: Path) -> tuple[pd.DataFrame, pd.Series]:
    data = pd.read_csv(csv_path)
    print("Raw Data:\n", data)

    # The customer id is an identifier, not a predictive signal.
    data = data.drop(columns=["customerID"])

    # Some TotalCharges values are stored as text and may contain blanks.
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

    print("Data Numericalize Churn:\n", data)

    features = data.drop(columns=["Churn"])
    target = data["Churn"]
    return features, target


def attach_training_monitoring(model: Pipeline) -> tuple[object | None, str]:
    scoring_monitor = None
    monitoring_mode = "verbose-only"

    try:
        scoring_monitor = ScoringMonitor(scoring={"accuracy": "accuracy"})
    except Exception:
        return None, monitoring_mode

    try:
        model.set_callbacks(scoring_monitor)
        monitoring_mode = "scoring-only"
    except Exception:
        return None, monitoring_mode

    try:
        model.set_callbacks(ProgressBar(), scoring_monitor)
        monitoring_mode = "progress-bar-and-scoring"
    except Exception:
        # Some scikit-learn callback combinations are not supported on Pipeline.
        pass

    return scoring_monitor, monitoring_mode


def build_model(
    numeric_features: list[str],
    categorical_features: list[str],
    categorical_encoding: str,
    numeric_scaler: str,
    enable_monitoring: bool = True,
) -> Pipeline:
    # Imputer defines what value to use for missing inputs.
    # Scaler defines how to rescale the data before training.
    if numeric_scaler == "standard":
        scaler = StandardScaler()
    else:
        scaler = BadMagnitudeScaler()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
        ]
    )

    if categorical_encoding == "onehot":
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        # This is deliberately simplistic so learners can see why ordinal encoding can mislead.
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1,
        )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", encoder),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    # f(w, X[k]) = sum_i w[i] * X[k][i]
    # p(w, X[k]) = exp(f(w, X[k])) / (exp(f(w, X[k])) + 1)
    # loss(w) = sum_k {-y[k] ln p(w, X[k]) - (1-y[k]) ln (1-p(w, X[k]))}
    classifier = LogisticRegression(max_iter=10, solver="lbfgs", verbose=1)
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )

    if enable_monitoring:
        scoring_monitor, monitoring_mode = attach_training_monitoring(model)
    else:
        scoring_monitor, monitoring_mode = None, "verbose-only"

    model._scoring_monitor = scoring_monitor
    model._fit_monitoring_mode = monitoring_mode
    model._categorical_encoding = categorical_encoding
    model._numeric_scaler = numeric_scaler
    return model


In [6]:
# ==================
# Train and evaluate
# ==================
X, y = load_data(config.data)

print("X:\n", X)
print("y:\n", y)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("numeric_features:", numeric_features)
print("categorical_features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config.test_size,
    random_state=config.random_state,
    stratify=y,
)

model = build_model(
    numeric_features,
    categorical_features,
    config.categorical_encoding,
    config.numeric_scaler,
)

print(f"Fit monitoring mode: {model._fit_monitoring_mode}")
print(f"Categorical encoding mode: {config.categorical_encoding}")
print(f"Numeric scaler mode: {config.numeric_scaler}")

try:
    model.fit(X_train, y_train)
except TypeError as error:
    if "auto-propagated callbacks" not in str(error):
        raise

    print(
        "Pipeline callbacks are not compatible with this scikit-learn build; "
        "retrying with verbose-only monitoring."
    )
    model = build_model(
        numeric_features,
        categorical_features,
        config.categorical_encoding,
        config.numeric_scaler,
        enable_monitoring=False,
    )
    model.fit(X_train, y_train)

show_preprocessing_preview(
    model=model,
    X_train=X_train,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    categorical_encoding=config.categorical_encoding,
    numeric_scaler=config.numeric_scaler,
    preview_rows=config.preview_rows,
    preprocessed_out=config.preprocessed_out,
)

classifier = model.named_steps["classifier"]
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Solver iterations used: {classifier.n_iter_}")
print(f"Accuracy: {accuracy:.4f}")

monitor = getattr(model, "_scoring_monitor", None)
if monitor is not None:
    try:
        score_log = monitor.get_logs().data_as_pandas
        score_log = score_log.loc[
            score_log["accuracy"].notna(),
            ["task_name", "task_id", "accuracy"],
        ]
        if not score_log.empty:
            if len(score_log) > 10:
                score_log = score_log.tail(10)
            print("\nTraining accuracy snapshots:")
            print(score_log.to_string(index=False))
    except ValueError:
        pass

print("\nClassification report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=["No Churn", "Churn"],
    )
)

with config.model_out.open("wb") as model_file:
    pickle.dump(model, model_file)
print(f"Saved trained pipeline to: {config.model_out}")


FileNotFoundError: [Errno 2] No such file or directory: 'WA_Fn-UseC_-Telco-Customer-Churn.csv'